# **데이터 집계 심화**(Advanced Data Aggregation)

> **groupby에서 시작해 pivot까지** — 분석가의 손에 데이터를 길들이는 법

---

## 학습 목표

본 차시를 마치면 다음을 할 수 있다.

1. **`groupby`**(그룹별 집계)를 자유롭게 사용하여 단일/다중 그룹 통계를 계산한다.
2. **`crosstab`**(교차표)으로 범주형 변수 간의 관계를 빈도와 비율로 표현한다.
3. **`pivot_table`**(피벗테이블)을 활용해 다축 요약 분석을 수행한다.
4. **`stack`**/**`unstack`**/**`melt`**(데이터 형태 변환)을 사용해 wide↔long 형식을 자유롭게 오간다.

---




## 진행 방식

**Pong**(학생 실습)의 반복 구조로 진행된다. 각 셀의 코드를 직접 실행해 보고, 빈칸을 채워야 한다. 정답은 각 문제 아래 토글 버튼을 눌러 확인할 수 있다.

> → **권고**: 빈칸을 먼저 직접 채워본 후에 정답을 확인하는 습관을 기르길 바란다. 학습 효과가 크게 달라진다.


## **0. 환경 설정**(Environment Setup)

분석에 필요한 라이브러리를 불러오고 데이터를 로드한다. 본 차시에서는 다음 라이브러리를 사용한다.

- **pandas**(판다스): 표 형식 데이터 처리
- **numpy**(넘파이): 수치 연산
- **matplotlib**/**seaborn**: 시각화 (1차시 후반부에 일부 사용)

---

### 라이브러리 import


In [8]:
# 필수 라이브러리 import
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 화면 출력 설정
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print('pandas 버전:', pd.__version__)
print('numpy  버전:', np.__version__)
print('환경 설정 완료')

pandas 버전: 2.2.2
numpy  버전: 2.0.2
환경 설정 완료


### 데이터 로드


In [9]:
# Ping 데이터: Airbnb NYC 2019
URL_AIRBNB = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AB_NYC_2019.csv"

try:
    airbnb = pd.read_csv(URL_AIRBNB)
    print(f'Airbnb 로드 성공: {airbnb.shape[0]:,} 행 × {airbnb.shape[1]} 열')
except Exception:
    # 폴백: 합성 데이터 생성 (네트워크 차단 시)
    import random
    random.seed(42)
    np.random.seed(42)
    n = 5000
    boroughs = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
    room_types = ['Entire home/apt', 'Private room', 'Shared room']
    airbnb = pd.DataFrame({
        'id': range(1, n+1),
        'name': [f'Listing_{i}' for i in range(1, n+1)],
        'host_id': np.random.randint(1, 2000, n),
        'neighbourhood_group': np.random.choice(boroughs, n, p=[0.45, 0.35, 0.12, 0.05, 0.03]),
        'neighbourhood': np.random.choice(['Midtown', 'Harlem', 'Williamsburg', 'Astoria', 'Flushing'], n),
        'latitude': np.random.uniform(40.5, 40.9, n),
        'longitude': np.random.uniform(-74.2, -73.7, n),
        'room_type': np.random.choice(room_types, n, p=[0.5, 0.45, 0.05]),
        'price': np.random.gamma(2, 80, n).astype(int) + 20,
        'minimum_nights': np.random.choice([1, 2, 3, 5, 7, 30], n, p=[0.4, 0.25, 0.15, 0.1, 0.05, 0.05]),
        'number_of_reviews': np.random.poisson(20, n),
        'reviews_per_month': np.round(np.random.gamma(1.5, 1, n), 2),
        'calculated_host_listings_count': np.random.choice([1, 2, 3, 5, 10], n, p=[0.7, 0.15, 0.08, 0.05, 0.02]),
        'availability_365': np.random.randint(0, 366, n),
    })
    print(f'Airbnb 합성데이터 생성: {airbnb.shape[0]:,} 행 × {airbnb.shape[1]} 열')

airbnb.head(3)

Airbnb 로드 성공: 48,895 행 × 16 열


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365


In [24]:
import pandas as pd
from io import StringIO
from urllib.request import Request, urlopen

# GitHub raw CSV 주소
url = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/compact.csv"

# 1. CSV 원문 다운로드
req = Request(url, headers={"User-Agent": "Mozilla/5.0"})

with urlopen(req) as response:
    raw_text = response.read().decode("utf-8-sig")

# 2. 각 줄 전체를 감싸는 큰따옴표 제거
# 예: "Afghanistan,2020-01-01,..."  ->  Afghanistan,2020-01-01,...
fixed_text = "\n".join(
    line[1:-1] if len(line) >= 2 and line.startswith('"') and line.endswith('"') else line
    for line in raw_text.splitlines()
)

# 3. pandas DataFrame으로 읽기
covid = pd.read_csv(
    StringIO(fixed_text),
    sep=",",
    na_values=[""],
    keep_default_na=True,
    parse_dates=["date"],
    low_memory=False
)

# 4. 확인
print(covid.shape)
print(covid.head())

print(covid.columns.tolist())
print(covid[["country", "date", "total_cases", "new_cases", "code", "continent"]].head())
print(covid.columns)


(59471, 61)
       country       date  total_cases  new_cases  new_cases_smoothed  total_cases_per_million  new_cases_per_million  new_cases_smoothed_per_million  total_deaths  new_deaths  new_deaths_smoothed  \
0  Afghanistan 2020-01-01          NaN        NaN                 NaN                      NaN                    NaN                             NaN           NaN         NaN                  NaN   
1  Afghanistan 2020-01-02          NaN        NaN                 NaN                      NaN                    NaN                             NaN           NaN         NaN                  NaN   
2  Afghanistan 2020-01-03          NaN        NaN                 NaN                      NaN                    NaN                             NaN           NaN         NaN                  NaN   
3  Afghanistan 2020-01-04          0.0        0.0                 NaN                      0.0                    0.0                             NaN           0.0         0.0             

---

# **Part 1. `groupby` — 그룹별 집계의 모든 것**

## **1.1 왜 `groupby`가 중요한가?**

데이터 분석에서 가장 자주 던지는 질문은 다음과 같다.

> **"카테고리별로** 평균이 어떻게 다른가?**"**
> **"지역별로** 매출이 얼마나 차이나는가?**"**
> **"월별로** 신규 가입자가 얼마나 늘었는가?**"**

이 질문들의 공통점은 **"~별로 ~을 구하라"** 형태라는 것이다. 이를 코드로 옮기는 도구가 바로 **`groupby`**(그룹화)이다.

---

### **`groupby`의 작동 원리** — Split-Apply-Combine 패러다임

```
[원본 데이터]
     │
     ▼
  ① Split   (분할)    : 키 컬럼을 기준으로 데이터를 그룹별로 나눈다.
     │
     ▼
  ② Apply   (적용)    : 각 그룹에 통계 함수(mean, sum, count 등)를 적용한다.
     │
     ▼
  ③ Combine (결합)    : 그룹별 결과를 하나의 표로 합친다.
```

이 세 단계를 **`pandas`**가 한 줄로 수행한다.

> **AI 연결**: 머신러닝의 **클래스별 통계량 계산**, 추천 시스템의 **사용자 클러스터별 선호도 집계** 등이 모두 `groupby`의 응용이다. 또한 **PyTorch**의 `scatter_reduce` 연산도 본질적으로는 같은 분할-집계 패러다임이다.


## **1.2 단일 그룹 집계**

### Ping 1 — 자치구(borough)별 평균 가격

뉴욕시는 5개 자치구(Manhattan, Brooklyn, Queens, Bronx, Staten Island)로 구성된다. 자치구별 Airbnb 평균 가격을 구해보자.

In [10]:
# Ping 1: 자치구별 평균 가격
result = airbnb.groupby('neighbourhood_group')['price'].mean()
print(result)
print()
print('타입:', type(result))

neighbourhood_group
Bronx             87.496792
Brooklyn         124.383207
Manhattan        196.875814
Queens            99.517649
Staten Island    114.812332
Name: price, dtype: float64

타입: <class 'pandas.core.series.Series'>


### **결과 해석**

- 결과는 **`Series`**(시리즈)이다. 인덱스는 자치구 이름, 값은 평균 가격이다.
- 단일 컬럼만 집계할 때는 자동으로 Series가 반환된다.
- 여러 컬럼을 동시에 집계하려면 컬럼명을 리스트로 전달한다 (다음 셀 참고).


In [12]:
# 여러 컬럼 동시 집계 → DataFrame 반환
result_multi = airbnb.groupby('neighbourhood_group')[['price', 'number_of_reviews']].mean()
print(result_multi)
print()
print('타입:', type(result_multi))

                          price  number_of_reviews
neighbourhood_group                               
Bronx                 87.496792          26.004583
Brooklyn             124.383207          24.202845
Manhattan            196.875814          20.985596
Queens                99.517649          27.700318
Staten Island        114.812332          30.941019

타입: <class 'pandas.core.frame.DataFrame'>


### ✎ 개념 확인 1

다음 코드의 결과 타입은 무엇인가?

```python
airbnb.groupby('neighbourhood_group')[['price']].mean()
```

<details><summary>▶ 정답 보기</summary>

**`DataFrame`**(데이터프레임)이다. 컬럼명을 **리스트로 감싸면**(`[['price']]`) 결과가 항상 DataFrame이 된다. 반면 단일 문자열(`['price']`)을 사용하면 Series가 된다.

이는 **`numpy`**의 1D 배열 vs 2D 배열의 구분과 유사하다. 머신러닝에서 입력을 항상 2D로 만드는 것(예: `X.reshape(-1, 1)`)과 동일한 사고방식이다.

</details>


### Pong 1 — 국가별 평균 확진자 수 (COVID-19 데이터)

COVID-19 데이터에서 **국가별** 평균 확진자 수(`Confirmed`)를 구하라.

> **힌트**: `covid.groupby('Country/Region')['Confirmed'].mean()`을 작성한다.

In [26]:
# Pong 1: 빈칸을 채워라
# (1) 'continent' 으로 그룹화
# (2) col 컬럼 선택
# (3) 평균 계산
cols = [
    "new_cases_per_million",
    "new_deaths_per_million",
    "new_cases_smoothed_per_million",
    "new_deaths_smoothed_per_million"
]

result_pong1 = covid.groupby(_____)[_____].mean()
print(continent_mean)

NameError: name '_____' is not defined

<details><summary>▶ 정답 보기</summary>

```python
result_pong1 = covid.groupby('Country/Region')['Confirmed'].mean()
print(result_pong1.head())
```

`groupby()`의 첫 번째 인자에는 **그룹 기준**(키 컬럼)을, 그 뒤 대괄호에는 **집계 대상 컬럼**을 지정한다.

</details>


## **1.3 다양한 집계 함수**

`mean()` 외에도 다양한 통계 함수를 적용할 수 있다.

| 함수 | 의미 |
|---|---|
| **`mean()`** | 평균 |
| **`median()`** | 중앙값 |
| **`sum()`** | 합계 |
| **`count()`** | 결측을 제외한 개수 |
| **`size()`** | 결측 포함 전체 개수 |
| **`min()`** / **`max()`** | 최솟값 / 최댓값 |
| **`std()`** / **`var()`** | 표준편차 / 분산 |
| **`first()`** / **`last()`** | 첫 / 마지막 값 |
| **`nunique()`** | 고유값 개수 |


### Ping 2 — 자치구별 다양한 통계

자치구별로 가격의 평균/중앙값/개수/표준편차를 모두 구해보자.

In [ ]:
# Ping 2: agg를 활용한 다중 통계
stats = airbnb.groupby('neighbourhood_group')['price'].agg(['mean', 'median', 'count', 'std'])
print(stats.round(2))

### **`agg()` 함수**(aggregate)의 위력

**`agg()`**는 한 번에 여러 통계량을 적용할 수 있는 강력한 메서드이다. 인자로 **함수명 문자열의 리스트**를 전달한다.

> **AI 연결**: 딥러닝 학습 로그를 분석할 때 에폭별 loss의 mean, min, max를 동시에 보는 작업이 정확히 이 패턴이다.

---

### Ping 3 — 컬럼별로 다른 함수 적용하기

가격은 평균을, 리뷰 수는 합계를 구하고 싶다면 어떻게 할까? **딕셔너리**를 활용한다.

In [ ]:
# Ping 3: 컬럼별 다른 함수 적용
result = airbnb.groupby('neighbourhood_group').agg({
    'price': 'mean',
    'number_of_reviews': 'sum',
    'minimum_nights': 'median',
})
print(result.round(2))

### ✎ 개념 확인 2

`groupby` 결과에서 **그룹 키 컬럼**(`neighbourhood_group`)이 인덱스가 되어 있다. 이를 일반 컬럼으로 되돌리려면 어떻게 해야 하는가?

<details><summary>▶ 정답 보기</summary>

**`reset_index()`** 메서드를 호출하면 된다.

```python
result.reset_index()
```

또는 `groupby()`에 **`as_index=False`** 옵션을 주면 처음부터 인덱스 대신 일반 컬럼으로 유지된다.

```python
airbnb.groupby('neighbourhood_group', as_index=False)['price'].mean()
```

</details>


### Pong 2 — 국가별 확진/사망/회복 통계

COVID-19 데이터에서 국가별로 다음 통계를 한 번에 구하라.

- `Confirmed`: 평균
- `Deaths`: 합계
- `Recovered`: 최댓값


In [ ]:
# Pong 2: agg + 딕셔너리

result_pong2 = covid.groupby('Country/Region').agg({
    'Confirmed': _____,
    'Deaths': _____,
    'Recovered': _____,
})
print(result_pong2.round(2))

<details><summary>▶ 정답 보기</summary>

```python
result_pong2 = covid.groupby('Country/Region').agg({
    'Confirmed': 'mean',
    'Deaths': 'sum',
    'Recovered': 'max',
})
print(result_pong2.round(2))
```

함수명을 **문자열로 전달**하는 점을 기억해 두어야 한다. 함수 객체(`np.mean`)도 가능하지만, 문자열이 가독성이 더 좋다.

</details>


## **1.4 다중 그룹 집계**

여러 컬럼을 동시에 그룹 키로 사용할 수 있다. 예를 들어 **자치구 × 방 유형**별 평균 가격을 보고 싶다면, 두 컬럼을 리스트로 전달한다.

### Ping 4 — 자치구 × 방 유형별 평균 가격

In [ ]:
# Ping 4: 다중 그룹화
result = airbnb.groupby(['neighbourhood_group', 'room_type'])['price'].mean()
print(result.round(1))
print()
print('인덱스 타입:', type(result.index).__name__)

### **`MultiIndex`**(다중 인덱스)의 등장

다중 그룹화의 결과는 **계층적 인덱스**(`MultiIndex`)를 가진다. 이는 데이터를 더 풍부하게 표현하지만, 다루는 데 익숙해질 시간이 필요하다.

---

### **계층 인덱스 시각화**

```
                            price
neighbourhood_group  room_type
Bronx                Entire home/apt    127.5
                     Private room        66.8
                     Shared room         59.8
Brooklyn             Entire home/apt    178.3
                     Private room        76.5
...                  ...                 ...
```

> **`unstack()`을 사용하면** 다중 인덱스를 컬럼으로 풀 수 있다 (Part 4에서 다룬다).


In [ ]:
# unstack() 미리보기 - 한쪽 인덱스를 컬럼으로 변환
result.unstack().round(1)

### Pong 3 — 국가 × 월별 사망자 수

COVID-19 데이터에서 **국가별 + 월별** 사망자(`Deaths`)의 합계를 구하라.

> **힌트**: 월을 추출하려면 `pd.to_datetime(covid['Date']).dt.month`를 새로운 컬럼으로 추가해야 한다.

In [ ]:
# Pong 3: 다중 그룹화
# (1) 'Date'에서 월 추출
covid['Month'] = pd.to_datetime(covid['Date']).dt.month

# (2) Country/Region + Month 로 그룹화하여 Deaths 합계
result_pong3 = covid.groupby([_____, _____])[_____].sum()
print(result_pong3.head(15))

<details><summary>▶ 정답 보기</summary>

```python
covid['Month'] = pd.to_datetime(covid['Date']).dt.month
result_pong3 = covid.groupby(['Country/Region', 'Month'])['Deaths'].sum()
print(result_pong3.head(15))
```

다중 그룹화의 핵심은 **그룹 키를 리스트로 묶는 것**이다. 결과는 자연스럽게 `MultiIndex` `Series`가 된다.

</details>


## **1.5 `transform`과 `filter` — 고급 그룹 연산**

`groupby`는 단순 집계만이 아니라, **그룹별 변환**과 **그룹 필터링**도 지원한다.

| 메서드 | 결과 길이 | 용도 |
|---|---|---|
| **`agg()`** | 그룹 수만큼 축소 | 그룹별 통계량 |
| **`transform()`** | 원본과 동일 | 그룹별 표준화·정규화 |
| **`filter()`** | 일부 행 제거 | 조건을 만족하는 그룹만 남기기 |

---

### Ping 5 — `transform`으로 그룹별 z-score 계산

자치구별로 가격을 표준화(z-score)해 보자. **각 행의 가격이 자치구 평균에서 몇 표준편차만큼 떨어져 있는가**를 계산한다.

In [ ]:
# Ping 5: groupby + transform
airbnb['price_zscore'] = (
    airbnb.groupby('neighbourhood_group')['price']
    .transform(lambda x: (x - x.mean()) / x.std())
)
print(airbnb[['neighbourhood_group', 'price', 'price_zscore']].head(10).round(3))

### **`transform`이 왜 강력한가?**

- `agg`는 그룹 수만큼 결과가 축소된다 (5개 자치구 → 5행).
- **`transform`**은 **원본 길이를 유지**한다 (49,000행 → 49,000행).
- 따라서 **머신러닝의 그룹별 정규화**(예: 사용자별 평점 정규화)에 즉시 활용할 수 있다.

> **AI 연결**: 추천 시스템에서 사용자별로 평점 스케일이 다를 때, `groupby('user_id')['rating'].transform(zscore)`로 정규화하는 것이 일반적이다. 또한 **딥러닝**의 **BatchNormalization**도 미니배치 단위로 같은 변환을 수행한다.

---

### Ping 6 — `filter`로 리뷰가 100건 이상인 자치구만 보기

자치구별 총 리뷰 수가 일정 기준을 넘는 자치구만 필터링한다.

In [ ]:
# Ping 6: groupby + filter
filtered = airbnb.groupby('neighbourhood_group').filter(
    lambda g: g['number_of_reviews'].sum() > 50000
)
print('필터링 전:', airbnb.shape)
print('필터링 후:', filtered.shape)
print('남은 자치구:', filtered['neighbourhood_group'].unique())

### ✎ 개념 확인 3

다음 두 코드의 차이를 설명하라.

```python
# (A)
airbnb.groupby('room_type')['price'].mean()

# (B)
airbnb.groupby('room_type')['price'].transform('mean')
```

<details><summary>▶ 정답 보기</summary>

- **(A)**는 `room_type`별 평균을 구한다. 결과는 **3행 Series**(방 유형 3종이라 가정)이다.
- **(B)**는 각 행에 그 행이 속한 방 유형의 평균을 채워 넣는다. 결과는 **원본과 동일한 길이의 Series**이다.

(B)는 **그룹 통계량을 원본 행 단위에 매핑**할 때 사용된다. 예를 들어 가격을 그룹 평균으로 나눈 비율(`price / group_mean`)을 만들 때 매우 유용하다.

</details>


---

## **1.6 연습문제 (10문항)**

다음 문제들은 모두 `airbnb` 또는 `covid` 데이터프레임을 사용한다. 각 문제 아래의 토글을 눌러 정답을 확인하라.

---

### **연습문제 1-1**

자치구별 **최대 가격**(`price`)을 구하라.

In [ ]:
# 연습문제 1-1
ans_1_1 = ____________________________________________
print(ans_1_1)

<details><summary>▶ 정답 보기</summary>

```python
ans_1_1 = airbnb.groupby('neighbourhood_group')['price'].max()
print(ans_1_1)
```
</details>

---

### **연습문제 1-2**

방 유형(`room_type`)별 **숙소 개수**를 구하라. 결과를 내림차순으로 정렬하라.

In [ ]:
# 연습문제 1-2
ans_1_2 = ____________________________________________
print(ans_1_2)

<details><summary>▶ 정답 보기</summary>

```python
ans_1_2 = airbnb.groupby('room_type').size().sort_values(ascending=False)
print(ans_1_2)
```

`size()`는 결측을 포함한 전체 개수를, `count()`는 결측을 제외한 개수를 반환한다.

</details>

---

### **연습문제 1-3**

자치구별 가격의 **평균과 중앙값**을 동시에 구하라.

In [ ]:
# 연습문제 1-3
ans_1_3 = ____________________________________________
print(ans_1_3.round(2))

<details><summary>▶ 정답 보기</summary>

```python
ans_1_3 = airbnb.groupby('neighbourhood_group')['price'].agg(['mean', 'median'])
print(ans_1_3.round(2))
```

</details>

---

### **연습문제 1-4**

자치구 × 방 유형별 **숙소 개수**를 구하라.

In [ ]:
# 연습문제 1-4
ans_1_4 = ____________________________________________
print(ans_1_4)

<details><summary>▶ 정답 보기</summary>

```python
ans_1_4 = airbnb.groupby(['neighbourhood_group', 'room_type']).size()
print(ans_1_4)
```

</details>

---

### **연습문제 1-5**

자치구별로 다음 세 가지 통계를 한 번에 구하라.

- 가격(`price`)의 평균
- 리뷰 수(`number_of_reviews`)의 합계
- 최소 숙박일(`minimum_nights`)의 중앙값


In [ ]:
# 연습문제 1-5
ans_1_5 = ____________________________________________
print(ans_1_5.round(2))

<details><summary>▶ 정답 보기</summary>

```python
ans_1_5 = airbnb.groupby('neighbourhood_group').agg({
    'price': 'mean',
    'number_of_reviews': 'sum',
    'minimum_nights': 'median',
})
print(ans_1_5.round(2))
```

</details>

---

### **연습문제 1-6**

자치구별 평균 가격을 **`as_index=False`**를 사용해 일반 컬럼 형태로 구하라.

In [ ]:
# 연습문제 1-6
ans_1_6 = ____________________________________________
print(ans_1_6)

<details><summary>▶ 정답 보기</summary>

```python
ans_1_6 = airbnb.groupby('neighbourhood_group', as_index=False)['price'].mean()
print(ans_1_6)
```

`as_index=False`를 사용하면 그룹 키가 인덱스로 가지 않고 일반 컬럼으로 남는다. 후속 처리(병합·필터링)가 편해진다.

</details>

---

### **연습문제 1-7**

각 행의 가격을 **자치구 평균으로 나눈 비율**을 구해 새 컬럼 `price_ratio`로 추가하라.

In [ ]:
# 연습문제 1-7
airbnb['price_ratio'] = ____________________________________________
print(airbnb[['neighbourhood_group', 'price', 'price_ratio']].head(10).round(3))

<details><summary>▶ 정답 보기</summary>

```python
airbnb['price_ratio'] = (
    airbnb['price'] /
    airbnb.groupby('neighbourhood_group')['price'].transform('mean')
)
print(airbnb[['neighbourhood_group', 'price', 'price_ratio']].head(10).round(3))
```

`transform('mean')`이 핵심이다. 각 행에 그 행이 속한 자치구의 평균이 채워지므로, 단순 나눗셈으로 비율을 얻을 수 있다.

</details>

---

### **연습문제 1-8**

방 유형이 `'Entire home/apt'`인 행만 골라, 자치구별 평균 가격을 구하라.

In [ ]:
# 연습문제 1-8
ans_1_8 = ____________________________________________
print(ans_1_8.round(2))

<details><summary>▶ 정답 보기</summary>

```python
ans_1_8 = (
    airbnb[airbnb['room_type'] == 'Entire home/apt']
    .groupby('neighbourhood_group')['price'].mean()
)
print(ans_1_8.round(2))
```

`groupby` 전에 **불리언 인덱싱**으로 행을 먼저 거르는 패턴이다. SQL의 `WHERE` 절과 같은 역할을 한다.

</details>

---

### **연습문제 1-9**

각 자치구에서 가격이 **상위 5번째 백분위수**(상위 5%)에 해당하는 가격 임계값을 구하라.

> **힌트**: 분위수는 `quantile(0.95)`로 구한다.


In [ ]:
# 연습문제 1-9
ans_1_9 = ____________________________________________
print(ans_1_9.round(2))

<details><summary>▶ 정답 보기</summary>

```python
ans_1_9 = airbnb.groupby('neighbourhood_group')['price'].quantile(0.95)
print(ans_1_9.round(2))
```

`quantile()`은 그룹별 분위수를 빠르게 계산한다. 머신러닝의 **이상치 탐지**에서 그룹별 임계값을 정할 때 빈번히 사용된다.

</details>

---

### **연습문제 1-10**

COVID-19 데이터에서 국가별로 **마지막 날짜의 누적 확진자 수**를 구하라.

> **힌트**: 그룹별 `last()`를 활용하거나, 정렬 후 `tail(1)`을 활용할 수 있다.


In [ ]:
# 연습문제 1-10
covid_sorted = covid.sort_values(['Country/Region', 'Date'])
ans_1_10 = ____________________________________________
print(ans_1_10.head(10))

<details><summary>▶ 정답 보기</summary>

```python
covid_sorted = covid.sort_values(['Country/Region', 'Date'])
ans_1_10 = covid_sorted.groupby('Country/Region')['Confirmed'].last()
print(ans_1_10.head(10))
```

또는:

```python
ans_1_10 = covid_sorted.groupby('Country/Region').tail(1).set_index('Country/Region')['Confirmed']
```

`last()`는 각 그룹의 **마지막 비결측 값**을 반환한다. 시계열에서 매우 유용하다.

</details>


---

# **Part 2. `crosstab` — 범주형 변수의 교차표**

## **2.1 교차표가 필요한 순간**

다음과 같은 질문에 답하려면 **두 범주형 변수의 결합 분포**가 필요하다.

> **"자치구별로** 어떤 방 유형이 가장 많은가?**"**
> **"성별과 흡연 여부**의 관계는 어떠한가?**"**
> **"질병 유무에 따른** 검사 결과의 분포는?**"**

이런 질문에 대한 답은 **교차표**(contingency table)로 표현된다. `pandas`는 이를 위해 **`pd.crosstab()`** 함수를 제공한다.

---

### **`crosstab` vs `groupby` — 무엇이 다른가?**

| 특성 | `groupby` | `crosstab` |
|---|---|---|
| **결과 형태** | 보통 긴(long) 형식 | 항상 와이드(wide) 형식 — 행과 열이 분리됨 |
| **주된 용도** | 임의의 집계 함수 | 빈도(또는 비율) 계산에 특화 |
| **마진(합계)** | 별도 계산 필요 | `margins=True`로 한 번에 |
| **정규화** | 직접 계산 | `normalize` 옵션 |

> **요약**: 두 범주의 **빈도/비율 표**를 만들 때는 `crosstab`이 가장 직관적이다. 통계학·의학 연구에서 자주 쓰인다.

---

### **AI 연결 — `crosstab`은 분류 모델의 평가 도구이다**

- 머신러닝의 **혼동행렬**(confusion matrix)은 **실제 클래스 × 예측 클래스**의 crosstab이다.
- **카이제곱 검정**(독립성 검정)의 입력 자료가 바로 crosstab이다.
- **나이브 베이즈** 분류기는 crosstab을 통해 **조건부 확률**을 계산한다.


## **2.2 기본 사용법 — 빈도 교차표**

### Ping 7 — 자치구 × 방 유형 빈도표

In [ ]:
# Ping 7: 가장 기본적인 crosstab
ct = pd.crosstab(airbnb['neighbourhood_group'], airbnb['room_type'])
print(ct)

### **결과 해석**

- **행**: 첫 번째 인자(`neighbourhood_group`)의 카테고리
- **열**: 두 번째 인자(`room_type`)의 카테고리
- **셀 값**: 두 카테고리의 조합에 해당하는 **행 개수**(빈도)

> 위 결과를 보면 Manhattan에서는 `Entire home/apt`가, Brooklyn에서는 `Private room`이 더 많을 수 있다. 자치구별 숙박 문화의 차이를 한눈에 볼 수 있다.

---

### Ping 8 — 행/열 합계 추가하기

In [ ]:
# Ping 8: margins=True로 합계 추가
ct = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    margins=True,         # 행/열 합계 추가
    margins_name='합계',  # 합계 이름 변경
)
print(ct)

### ✎ 개념 확인 4

`margins=True`로 추가된 마지막 행과 마지막 열은 무엇을 의미하는가?

<details><summary>▶ 정답 보기</summary>

- **마지막 행**: 각 방 유형의 전체 빈도 합계 (열 합계)
- **마지막 열**: 각 자치구의 전체 빈도 합계 (행 합계)
- **우측 하단 셀**: 전체 행의 개수 (총합)

이 값들은 **주변 분포**(marginal distribution)에 해당한다. 통계학에서 결합분포로부터 주변분포를 얻는 작업이 정확히 이것이다.

</details>


## **2.3 비율 교차표 — `normalize`**

빈도가 아닌 **비율**을 보고 싶다면 `normalize` 옵션을 사용한다.

| 옵션 | 의미 |
|---|---|
| **`normalize='all'`** | 전체에 대한 비율 — 모든 셀의 합이 1 |
| **`normalize='index'`** | 행별 비율 — 각 행의 합이 1 (행 = 100%) |
| **`normalize='columns'`** | 열별 비율 — 각 열의 합이 1 (열 = 100%) |

### Ping 9 — 행별 비율 교차표

각 자치구에서 방 유형의 분포가 어떻게 다른지 보고 싶다면 **`normalize='index'`**를 사용한다.

In [ ]:
# Ping 9: 행별 비율
ct_pct = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    normalize='index',
) * 100
print(ct_pct.round(1))

### **결과 해석**

각 행이 100%가 되도록 정규화된다. 예를 들어 Manhattan 행을 보면 `Entire home/apt`가 약 60%, `Private room`이 약 40% 등으로 표시된다 (실제 비율은 데이터에 따라 다름).

> **인사이트 도출의 핵심**: 절대 빈도만 보면 큰 자치구가 항상 우세해 보이지만, **행별 비율**로 보면 자치구마다의 **상대적 선호도**가 드러난다. 분석가는 이 두 시각을 모두 보여주어야 한다.

---

### Ping 10 — 한 번에 빈도와 비율 보기

세 줄로 작성해도 좋지만, 분석 결과 보고서에서는 두 표를 나란히 보여주는 것이 좋다.

In [ ]:
# Ping 10: 빈도 + 비율 동시 비교
print('=== 빈도 ===')
freq = pd.crosstab(airbnb['neighbourhood_group'], airbnb['room_type'])
print(freq)
print()
print('=== 행별 비율 (%) ===')
pct = pd.crosstab(airbnb['neighbourhood_group'], airbnb['room_type'], normalize='index') * 100
print(pct.round(1))

## **2.4 값을 추가한 교차표 — `values` + `aggfunc`**

`crosstab`은 기본적으로 빈도를 계산하지만, **`values`**와 **`aggfunc`** 옵션으로 다른 통계량을 계산할 수도 있다.

### Ping 11 — 자치구 × 방 유형별 평균 가격

In [ ]:
# Ping 11: values + aggfunc로 평균 가격 표 만들기
ct_price = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    values=airbnb['price'],
    aggfunc='mean',
)
print(ct_price.round(1))

### **`groupby + unstack` vs `crosstab`**

위 결과는 사실 다음과 동일하다.

```python
airbnb.groupby(['neighbourhood_group', 'room_type'])['price'].mean().unstack()
```

> **그렇다면 언제 어떤 것을 쓰는가?**
>
> - **빈도/비율 위주**, **마진 추가**, **간결한 코드**: `crosstab` 권장
> - **다양한 집계 함수 동시 적용**, 복잡한 변환: `groupby + unstack` 또는 `pivot_table` 권장


## **2.5 다중 인덱스/컬럼 교차표**

### Ping 12 — 다중 인덱스로 더 풍부한 표 만들기

자치구를 가격대와 함께 보고 싶다면 인덱스를 두 개로 지정한다.

In [ ]:
# Ping 12: 가격대 컬럼 만들기
airbnb['price_band'] = pd.cut(
    airbnb['price'],
    bins=[0, 100, 200, 500, 100000],
    labels=['저가', '중가', '고가', '럭셔리'],
)

# 자치구 + 가격대 → 방 유형 교차표
ct_multi = pd.crosstab(
    [airbnb['neighbourhood_group'], airbnb['price_band']],
    airbnb['room_type'],
)
print(ct_multi.head(15))

### **`pd.cut`이란?**

연속형 수치를 **구간으로 나누어 범주로 변환**하는 함수이다. 머신러닝의 **이산화**(discretization, binning)에 해당한다.

- `bins`: 경계값 리스트
- `labels`: 각 구간의 이름

> **AI 연결**: 의사결정 트리나 LightGBM의 내부 동작이 이 이산화와 본질적으로 같다. 또한 연령을 '청년/중년/노년'으로 나누는 것 같은 **피처 엔지니어링**에서 빈번히 사용된다.

---

### ✎ 개념 확인 5

다음 코드의 의미를 설명하라.

```python
pd.crosstab(airbnb['room_type'], airbnb['neighbourhood_group'], normalize='columns') * 100
```

<details><summary>▶ 정답 보기</summary>

**각 자치구**에서 방 유형의 분포를 백분율로 보여준다. 행은 방 유형, 열은 자치구이며, **각 열(자치구)의 합이 100%**가 된다.

예를 들어 Manhattan 열은 그 자치구 내에서 방 유형이 어떻게 분포하는지를 의미한다. 이 시각은 **자치구를 "관찰 단위"로 두고** 그 내부의 방 유형 비율을 비교하는 데 적합하다.

</details>


---

## **2.6 연습문제 (10문항)**

---

### **연습문제 2-1**

`room_type`별 빈도를 `crosstab`으로 구하라.

> **힌트**: 단일 변수의 빈도는 두 번째 인자에 같은 변수를 넣거나 `value_counts()`를 사용한다. 여기서는 첫 번째 인자에 `room_type`, 두 번째 인자에 `'count'`로 구분되는 가짜 시리즈를 넣을 수도 있지만, 가장 단순한 방법은 `value_counts()`이다.

In [ ]:
# 연습문제 2-1
ans_2_1 = airbnb['room_type'].value_counts()
print(ans_2_1)

<details><summary>▶ 정답 보기</summary>

`value_counts()`가 가장 간단하다. crosstab을 굳이 쓰자면 다음과 같이 한다.

```python
pd.crosstab(index=airbnb['room_type'], columns='count')
```

`columns='count'`는 컬럼 이름이 'count' 하나뿐인 단일 컬럼 표를 만든다.

</details>

---

### **연습문제 2-2**

자치구 × 방 유형 교차표를 **전체 비율**(`normalize='all'`)로 구하라.

In [ ]:
# 연습문제 2-2
ans_2_2 = ____________________________________________
print(ans_2_2.round(3))

<details><summary>▶ 정답 보기</summary>

```python
ans_2_2 = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    normalize='all',
)
print(ans_2_2.round(3))
```

모든 셀의 합이 1이 된다. 곱하기 100을 하면 백분율이 된다.

</details>

---

### **연습문제 2-3**

자치구 × 방 유형의 **합계 행/열을 포함**한 빈도 교차표를 만들라. 합계 라벨은 `'Total'`로 한다.

In [ ]:
# 연습문제 2-3
ans_2_3 = ____________________________________________
print(ans_2_3)

<details><summary>▶ 정답 보기</summary>

```python
ans_2_3 = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    margins=True,
    margins_name='Total',
)
print(ans_2_3)
```

</details>

---

### **연습문제 2-4**

자치구 × 방 유형별 **평균 리뷰 수**(`number_of_reviews`)를 구하라.

In [ ]:
# 연습문제 2-4
ans_2_4 = ____________________________________________
print(ans_2_4.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_2_4 = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    values=airbnb['number_of_reviews'],
    aggfunc='mean',
)
print(ans_2_4.round(1))
```

`values`와 `aggfunc`을 함께 사용하면 빈도가 아닌 **다른 통계량**을 셀 값으로 채울 수 있다.

</details>

---

### **연습문제 2-5**

자치구 × 방 유형별 **가격의 합계와 중앙값**을 동시에 보고 싶다. `crosstab`만으로는 한 번에 두 통계를 못 구한다. **`groupby + agg + unstack`**으로 해결하라.

In [ ]:
# 연습문제 2-5
ans_2_5 = ____________________________________________
print(ans_2_5.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_2_5 = (
    airbnb.groupby(['neighbourhood_group', 'room_type'])['price']
    .agg(['sum', 'median'])
    .unstack(level='room_type')
)
print(ans_2_5.round(1))
```

복합 통계를 wide 형식으로 만들 때는 **`groupby + unstack`** 조합이 강력하다. 컬럼이 자동으로 MultiIndex가 된다.

</details>

---

### **연습문제 2-6**

가격대(`price_band`) × 방 유형의 **빈도와 행별 비율을 함께** 출력하라.

In [ ]:
# 연습문제 2-6
freq = pd.crosstab(airbnb['price_band'], airbnb['room_type'])
pct = ____________________________________________

print('=== 빈도 ===')
print(freq)
print()
print('=== 행별 비율 (%) ===')
print(pct.round(1))

<details><summary>▶ 정답 보기</summary>

```python
freq = pd.crosstab(airbnb['price_band'], airbnb['room_type'])
pct = pd.crosstab(airbnb['price_band'], airbnb['room_type'], normalize='index') * 100

print('=== 빈도 ===')
print(freq)
print()
print('=== 행별 비율 (%) ===')
print(pct.round(1))
```

</details>

---

### **연습문제 2-7**

자치구 × 방 유형별 **최대 가격**을 구하라.

In [ ]:
# 연습문제 2-7
ans_2_7 = ____________________________________________
print(ans_2_7)

<details><summary>▶ 정답 보기</summary>

```python
ans_2_7 = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    values=airbnb['price'],
    aggfunc='max',
)
print(ans_2_7)
```

</details>

---

### **연습문제 2-8**

자치구별로 가격대(`price_band`)의 분포를 행별 비율(%)로 구하라.

In [ ]:
# 연습문제 2-8
ans_2_8 = ____________________________________________
print(ans_2_8.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_2_8 = pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['price_band'],
    normalize='index',
) * 100
print(ans_2_8.round(1))
```

이 표는 **각 자치구에서 가격대가 어떻게 분포하는지**를 보여준다. Manhattan은 고가/럭셔리 비율이, Bronx는 저가/중가 비율이 높을 것으로 예상된다.

</details>

---

### **연습문제 2-9**

`covid` 데이터에서 **국가 × 월별** 누적 사망자 수의 합계 교차표를 만들라.

In [ ]:
# 연습문제 2-9 (covid['Month']는 Part 1에서 이미 만들었음)
ans_2_9 = ____________________________________________
print(ans_2_9.head(10))

<details><summary>▶ 정답 보기</summary>

```python
ans_2_9 = pd.crosstab(
    covid['Country/Region'],
    covid['Month'],
    values=covid['Deaths'],
    aggfunc='sum',
)
print(ans_2_9.head(10))
```

</details>

---

### **연습문제 2-10**

자치구 × 방 유형 빈도 교차표를 **`heatmap`**으로 시각화하라.

> **힌트**: `import seaborn as sns`를 사용하고, `sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd')`처럼 작성한다.

In [ ]:
# 연습문제 2-10
import matplotlib.pyplot as plt
import seaborn as sns

ct = pd.crosstab(airbnb['neighbourhood_group'], airbnb['room_type'])

plt.figure(figsize=(8, 5))
sns.heatmap(_____, annot=_____, fmt=_____, cmap=_____)
plt.title('자치구 × 방 유형 빈도', fontsize=13)
plt.tight_layout()
plt.show()

<details><summary>▶ 정답 보기</summary>

```python
import matplotlib.pyplot as plt
import seaborn as sns

ct = pd.crosstab(airbnb['neighbourhood_group'], airbnb['room_type'])

plt.figure(figsize=(8, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd')
plt.title('자치구 × 방 유형 빈도', fontsize=13)
plt.tight_layout()
plt.show()
```

- `annot=True`: 셀에 숫자 표시
- `fmt='d'`: 정수 형식 (소수는 `'.1f'` 등)
- `cmap='YlOrRd'`: 노랑→주황→빨강 색상 맵

> **시각화 팁**: 빈도 교차표를 히트맵으로 바꾸면 **수치 차이가 시각적으로 즉시 이해**된다. 큰 표일수록 효과가 크다.

</details>


---

# **Part 3. `pivot_table` — 다축 요약 분석의 표준**

## **3.1 `pivot_table`의 위치**

- **`groupby`**: 가장 일반적이고 유연하다. long 형식 결과.
- **`crosstab`**: 빈도/비율 표에 특화. wide 형식.
- **`pivot_table`**: **다축 요약 + 다중 통계 + 마진 + 결측 처리**를 하나로 묶은 통합 도구.

> 엑셀의 **피벗테이블**(Pivot Table)을 사용해 본 적이 있다면, `pd.pivot_table`은 그것의 파이썬 버전이라 보면 된다.

---

### **`pivot_table`의 인자 구조**

```python
pd.pivot_table(
    data,                # 데이터프레임
    index=...,           # 행 위치에 둘 컬럼
    columns=...,         # 열 위치에 둘 컬럼
    values=...,          # 셀 값으로 채울 컬럼
    aggfunc=...,         # 집계 함수
    fill_value=...,      # 결측 셀 채울 값
    margins=...,         # 합계 행/열 추가
)
```

| 옵션 | 역할 |
|---|---|
| **`index`** | 행 차원 |
| **`columns`** | 열 차원 |
| **`values`** | 셀에 채울 수치 |
| **`aggfunc`** | 어떻게 집계할지 (기본 `mean`) |
| **`fill_value`** | NaN을 대체할 값 (예: 0) |

---

### **AI 연결**

- **딥러닝 실험 결과 정리**: 학습률(row) × 배치사이즈(col) → 정확도(value) 매트릭스를 만드는 작업이 정확히 `pivot_table`이다.
- **추천 시스템**: 사용자(row) × 아이템(col) → 평점(value) 매트릭스를 만드는 작업도 동일하다. 이를 **사용자-아이템 행렬**(user-item matrix)이라 부른다.


## **3.2 단일 인덱스, 단일 컬럼**

### Ping 13 — 자치구 × 방 유형별 평균 가격

In [ ]:
# Ping 13: 가장 기본적인 pivot_table
pt = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc='mean',
)
print(pt.round(1))

### **`crosstab`과 거의 동일하지만 더 명시적이다**

위 결과는 다음과 동일하다.

```python
pd.crosstab(
    airbnb['neighbourhood_group'],
    airbnb['room_type'],
    values=airbnb['price'],
    aggfunc='mean',
)
```

차이점은 다음과 같다.

- **`pivot_table`**: `data` 인자로 데이터프레임을 한 번만 넘기면 된다. 코드가 더 깔끔하다.
- **`crosstab`**: 각 컬럼을 시리즈로 따로 넘긴다. 빈도 표에 특화.

> **권고**: 빈도/비율은 `crosstab`, 다른 통계는 `pivot_table`로 사용하는 것이 일반적이다.


## **3.3 다중 인덱스 / 다중 컬럼 / 다중 값**

`pivot_table`의 진짜 강점은 **세 가지를 모두 다중으로 지정**할 수 있다는 점이다.

### Ping 14 — 다중 컬럼 + 다중 값

In [ ]:
# Ping 14: 가격, 리뷰 수를 동시에 — 평균
pt = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values=['price', 'number_of_reviews'],
    aggfunc='mean',
)
print(pt.round(1))

### **결과 해석**

- 컬럼이 **2층 MultiIndex**가 된다.
- 1층: 측정 변수(`price`, `number_of_reviews`)
- 2층: 방 유형(`Entire home/apt`, `Private room`, `Shared room`)

> **분석가 팁**: 보고서에 그대로 붙여넣어도 충분히 이해된다. 다만 화면에서 표가 옆으로 길어지므로, 출판물에서는 한 번에 한 가지 측정만 보여주는 것이 좋다.


### Ping 15 — 다중 인덱스

자치구를 더 잘게 쪼개서, **자치구 + 가격대**를 행으로 두고 방 유형을 열로 두자.

In [ ]:
# Ping 15: 다중 인덱스
pt = pd.pivot_table(
    airbnb,
    index=['neighbourhood_group', 'price_band'],  # 2층 인덱스
    columns='room_type',
    values='price',
    aggfunc='mean',
)
print(pt.round(1).head(15))

## **3.4 다중 집계 함수**

각 셀에 **여러 통계량**을 동시에 채울 수 있다.

### Ping 16 — 평균 + 중앙값 + 개수

In [ ]:
# Ping 16: 다중 집계
pt = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc=['mean', 'median', 'count'],
)
print(pt.round(1).head())

### **컬럼별로 다른 집계 함수**

`aggfunc`에 **딕셔너리**를 전달하면 컬럼별로 다른 집계를 적용할 수 있다.

In [ ]:
# Ping 17: 컬럼별 다른 집계 함수
pt = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values=['price', 'number_of_reviews'],
    aggfunc={
        'price': 'mean',           # 가격은 평균
        'number_of_reviews': 'sum',  # 리뷰 수는 합계
    },
)
print(pt.round(1))

### ✎ 개념 확인 6

`pivot_table`을 사용했을 때, 어떤 그룹 조합에 데이터가 한 건도 없으면 셀에 무엇이 채워지는가? 이를 어떻게 처리할 수 있는가?

<details><summary>▶ 정답 보기</summary>

기본적으로 **`NaN`**(결측값)이 채워진다. **`fill_value=0`** 또는 다른 적절한 값을 지정하면 결측 셀을 자동으로 채울 수 있다.

```python
pd.pivot_table(..., fill_value=0)
```

다만 **빈도 분석**에서는 0으로 채우는 것이 자연스럽지만, **평균 분석**에서는 0이 의미를 왜곡할 수 있으므로 주의해야 한다 (없는 그룹의 "평균은 0"이 아니라 "정의되지 않음"이 정확하다).

</details>


## **3.5 `margins`와 `fill_value`의 활용**

### Ping 18 — 마진 + 결측 채우기 + 다중 집계

In [ ]:
# Ping 18: 종합 활용
pt = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc='mean',
    margins=True,         # 합계 행/열 추가
    margins_name='전체',  # 합계 라벨
    fill_value=0,         # 결측 → 0
)
print(pt.round(1))

### **마진의 의미**

- **마지막 행**: 각 방 유형의 **전체 평균 가격**
- **마지막 열**: 각 자치구의 **전체 평균 가격**
- **우측 하단 셀**: 전체 데이터의 평균 가격

> **주의**: `margins`의 합계 행은 **단순 산술 평균**이 아니라 **가중 평균**으로 계산된다. 그룹 크기가 다르므로 단순 평균과 결과가 다르다는 점을 기억해 두어야 한다.


## **3.6 `pivot` vs `pivot_table` — 무엇이 다른가?**

| 메서드 | 집계 가능? | 중복 키 허용? |
|---|---|---|
| **`df.pivot()`** | 불가능 | 불가능 (중복 시 에러) |
| **`pd.pivot_table()`** | 가능 (`aggfunc`) | 가능 |

### Ping 19 — `pivot`의 한계

In [ ]:
# Ping 19: 작은 예제로 차이 확인
demo = pd.DataFrame({
    'date': ['2024-01-01', '2024-01-01', '2024-01-02', '2024-01-02'],
    'product': ['A', 'B', 'A', 'B'],
    'sales': [100, 200, 150, 250],
})
print(demo)
print()

# pivot은 중복 없이 1:1 매핑만 가능
pivoted = demo.pivot(index='date', columns='product', values='sales')
print(pivoted)

In [ ]:
# 만약 같은 날 같은 제품이 두 번 나오면 pivot은 실패한다 (예외 발생)
demo2 = pd.DataFrame({
    'date': ['2024-01-01', '2024-01-01', '2024-01-01'],
    'product': ['A', 'A', 'B'],  # A가 두 번
    'sales': [100, 120, 200],
})

try:
    demo2.pivot(index='date', columns='product', values='sales')
except Exception:
    print('pivot 실패 → 중복 키 때문에 에러 발생')

# pivot_table은 aggfunc로 자동 집계
result = demo2.pivot_table(index='date', columns='product', values='sales', aggfunc='sum')
print()
print('pivot_table 결과:')
print(result)

### **결론**

- 데이터가 **이미 그룹별로 유일**하다면 `pivot`이 더 빠르고 명시적이다.
- 데이터에 **중복 키가 있을 가능성**이 있거나 **집계가 필요**하다면 `pivot_table`을 사용한다.
- 실무에서는 **`pivot_table`이 훨씬 안전한 기본값**이다.


---

## **3.7 연습문제 (10문항)**

---

### **연습문제 3-1**

자치구별 평균 **숙박 일수**(`minimum_nights`)를 `pivot_table`로 구하라.

> **힌트**: 단일 인덱스만 있고 columns는 없는 경우, columns 인자를 생략한다.

In [ ]:
# 연습문제 3-1
ans_3_1 = ____________________________________________
print(ans_3_1.round(2))

<details><summary>▶ 정답 보기</summary>

```python
ans_3_1 = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    values='minimum_nights',
    aggfunc='mean',
)
print(ans_3_1.round(2))
```

</details>

---

### **연습문제 3-2**

자치구 × 방 유형별 **가격의 합계**를 `pivot_table`로 구하라.

In [ ]:
# 연습문제 3-2
ans_3_2 = ____________________________________________
print(ans_3_2)

<details><summary>▶ 정답 보기</summary>

```python
ans_3_2 = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc='sum',
)
print(ans_3_2)
```

</details>

---

### **연습문제 3-3**

자치구 × 방 유형별 **평균과 중앙값**을 동시에 구하라.

In [ ]:
# 연습문제 3-3
ans_3_3 = ____________________________________________
print(ans_3_3.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_3_3 = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc=['mean', 'median'],
)
print(ans_3_3.round(1))
```

</details>

---

### **연습문제 3-4**

자치구 × 방 유형별 평균 가격에 **합계 행/열**을 추가하라.

In [ ]:
# 연습문제 3-4
ans_3_4 = ____________________________________________
print(ans_3_4.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_3_4 = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc='mean',
    margins=True,
    margins_name='전체',
)
print(ans_3_4.round(1))
```

</details>

---

### **연습문제 3-5**

가격대 × 방 유형별 **숙소 개수**를 `pivot_table`로 구하라. 결측은 0으로 채워라.

> **힌트**: 개수는 `aggfunc='size'` 또는 `aggfunc='count'`이다.

In [ ]:
# 연습문제 3-5
ans_3_5 = ____________________________________________
print(ans_3_5)

<details><summary>▶ 정답 보기</summary>

```python
ans_3_5 = pd.pivot_table(
    airbnb,
    index='price_band',
    columns='room_type',
    values='price',     # values는 아무거나
    aggfunc='size',     # 개수 집계
    fill_value=0,
)
print(ans_3_5)
```

`size`와 `count`의 차이를 다시 한번 짚어두자: **`size`는 결측 포함 전체**, **`count`는 결측 제외**이다.

</details>

---

### **연습문제 3-6**

자치구 × 방 유형별로 **가격의 평균**과 **리뷰 수의 합계**를 컬럼별로 다르게 집계하라.

In [ ]:
# 연습문제 3-6
ans_3_6 = ____________________________________________
print(ans_3_6.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_3_6 = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values=['price', 'number_of_reviews'],
    aggfunc={
        'price': 'mean',
        'number_of_reviews': 'sum',
    },
)
print(ans_3_6.round(1))
```

</details>

---

### **연습문제 3-7**

자치구 + 가격대를 인덱스로, 방 유형을 컬럼으로 하여 **평균 리뷰 수**를 구하라.

In [ ]:
# 연습문제 3-7
ans_3_7 = ____________________________________________
print(ans_3_7.round(1).head(10))

<details><summary>▶ 정답 보기</summary>

```python
ans_3_7 = pd.pivot_table(
    airbnb,
    index=['neighbourhood_group', 'price_band'],
    columns='room_type',
    values='number_of_reviews',
    aggfunc='mean',
)
print(ans_3_7.round(1).head(10))
```

</details>

---

### **연습문제 3-8**

`covid` 데이터에서 **국가 × 월별 누적 확진자 수의 평균**을 `pivot_table`로 구하라.

In [ ]:
# 연습문제 3-8
ans_3_8 = ____________________________________________
print(ans_3_8.round(0).head(10))

<details><summary>▶ 정답 보기</summary>

```python
ans_3_8 = pd.pivot_table(
    covid,
    index='Country/Region',
    columns='Month',
    values='Confirmed',
    aggfunc='mean',
)
print(ans_3_8.round(0).head(10))
```

</details>

---

### **연습문제 3-9**

자치구 × 방 유형별로 **상위 10% 가격대의 평균**을 구하라. 사용자 정의 함수로 `aggfunc`을 작성하라.

> **힌트**: `aggfunc=lambda s: s[s >= s.quantile(0.9)].mean()`

In [ ]:
# 연습문제 3-9
ans_3_9 = ____________________________________________
print(ans_3_9.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_3_9 = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc=lambda s: s[s >= s.quantile(0.9)].mean(),
)
print(ans_3_9.round(1))
```

`aggfunc`에 **임의의 함수**를 전달할 수 있다는 사실은 매우 강력하다. 이는 **함수형 프로그래밍**의 사고이며, **PyTorch의 사용자 정의 손실 함수**나 **scikit-learn의 사용자 정의 스코어러**와 같은 패턴이다.

</details>

---

### **연습문제 3-10**

자치구 × 방 유형별 평균 가격을 `pivot_table`로 구한 후, **각 자치구별로 어떤 방 유형이 가장 비싼지** 출력하라.

> **힌트**: 행별 최댓값의 컬럼명을 얻으려면 `pt.idxmax(axis=1)`을 사용한다.

In [ ]:
# 연습문제 3-10
pt = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc='mean',
)
ans_3_10 = ____________________________________________
print(ans_3_10)

<details><summary>▶ 정답 보기</summary>

```python
pt = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc='mean',
)
ans_3_10 = pt.idxmax(axis=1)
print(ans_3_10)
```

`idxmax`는 **최댓값의 인덱스**를 반환한다. `axis=1`은 행 단위로 최댓값을 찾는다는 의미이다. 머신러닝에서 **다중 클래스 분류 결과에서 가장 확률이 높은 클래스**를 뽑을 때(`np.argmax(probs, axis=1)`)와 동일한 사고이다.

</details>


---

# **Part 4. `stack` / `unstack` / `melt` — 데이터 형태 변환**

## **4.1 wide vs long — 두 가지 데이터 형식**

데이터프레임은 같은 정보라도 **두 가지 형태**로 표현될 수 있다.

---

### **wide format**(와이드 형식)

```
국가      |  2020년  |  2021년  |  2022년
---------+---------+---------+---------
한국      |   10    |   15    |   20
일본      |   12    |   18    |   25
```

- 사람이 보기 편하다 (보고서·엑셀 형태).
- 컬럼이 **카테고리 자체**를 의미한다.

---

### **long format**(롱 형식, "tidy data")

```
국가  |  연도   |  값
-----+--------+-----
한국  |  2020  |  10
한국  |  2021  |  15
한국  |  2022  |  20
일본  |  2020  |  12
일본  |  2021  |  18
일본  |  2022  |  25
```

- 머신이 처리하기 편하다 (시각화 라이브러리 입력 형태).
- **하나의 행 = 하나의 관찰**.

---

### **왜 둘 다 필요한가?**

| 상황 | 권장 형식 |
|---|---|
| 보고서, 표 인쇄 | **wide** |
| `seaborn`/`plotly`/`ggplot2` 시각화 | **long** |
| 회귀·머신러닝 모델링 | 보통 **long → 변환** 필요 |
| 통계 검정 | 경우에 따라 다름 |

---

### **변환 도구 매트릭스**

| 변환 방향 | pandas 메서드 |
|---|---|
| **wide → long** | `melt()`, `stack()` |
| **long → wide** | `pivot()`/`pivot_table()`, `unstack()` |

> **AI 연결**: 시계열 예측, 패널 회귀, 추천 시스템 등 거의 모든 ML 파이프라인의 **첫 단계**가 바로 wide↔long 변환이다. 이 과정을 자유롭게 다룰 수 있어야 비로소 데이터 분석가라 할 수 있다.


## **4.2 `stack` / `unstack` — 컬럼과 인덱스의 자리바꿈**

### **`stack`이 하는 일**: 컬럼을 인덱스로 내린다 (wide → long).
### **`unstack`이 하는 일**: 인덱스를 컬럼으로 올린다 (long → wide).

```
   wide                          long
┌──────┬──────┐               ┌─────┬─────┬─────┐
│  A   │  B   │   ─stack→     │ idx │ var │ val │
├──────┼──────┤               ├─────┼─────┼─────┤
│  10  │  20  │   ←unstack─   │  0  │  A  │ 10  │
│  30  │  40  │               │  0  │  B  │ 20  │
└──────┴──────┘               │  1  │  A  │ 30  │
                              │  1  │  B  │ 40  │
                              └─────┴─────┴─────┘
```

---

### Ping 20 — 작은 예제로 이해하기

In [ ]:
# Ping 20: stack / unstack 직관 잡기
demo = pd.DataFrame({
    'A': [10, 30],
    'B': [20, 40],
}, index=['x', 'y'])

print('=== 원본 (wide) ===')
print(demo)
print()

print('=== stack() 결과 (long, MultiIndex Series) ===')
stacked = demo.stack()
print(stacked)
print()

print('=== unstack() 결과 (다시 wide로) ===')
print(stacked.unstack())

### **결과 해석**

- `stack()`은 컬럼 라벨을 **인덱스의 한 층**으로 내렸다. 결과는 **`Series`**(단일 값 컬럼).
- `unstack()`은 그 반대 작업으로, 인덱스의 한 층을 **컬럼**으로 올렸다.
- **`stack`과 `unstack`은 정확히 서로의 역연산**이다.


### Ping 21 — `groupby`와 `unstack` 결합

`groupby`로 만든 MultiIndex Series를 와이드 표로 변환하는 가장 흔한 패턴이다.

In [ ]:
# Ping 21: groupby + unstack
result = airbnb.groupby(['neighbourhood_group', 'room_type'])['price'].mean()
print('=== groupby 결과 (long-ish, MultiIndex Series) ===')
print(result.head(10).round(1))
print()

print('=== unstack() 후 (wide) ===')
print(result.unstack().round(1))

### ✎ 개념 확인 7

`unstack(level=0)`과 `unstack(level=1)`의 차이를 설명하라.

<details><summary>▶ 정답 보기</summary>

- **`unstack(level=0)`**: **첫 번째 인덱스 층**을 컬럼으로 올린다.
- **`unstack(level=1)`** (기본값): **두 번째 인덱스 층**(가장 안쪽)을 컬럼으로 올린다.

위 Ping 21의 결과에서 `level=0`을 사용하면 **자치구가 컬럼**, **방 유형이 행**이 되고, `level=1`(기본)을 사용하면 그 반대가 된다.

또한 인덱스 이름으로도 지정 가능하다: `unstack(level='room_type')`. 가독성이 좋아 권장된다.

</details>


In [ ]:
# 비교 출력
result = airbnb.groupby(['neighbourhood_group', 'room_type'])['price'].mean()

print('=== unstack(level=0) → 자치구가 컬럼 ===')
print(result.unstack(level=0).round(1))
print()
print('=== unstack(level=1) → 방 유형이 컬럼 (기본값) ===')
print(result.unstack(level=1).round(1))

## **4.3 `melt` — 진짜 wide → long 변환의 정석**

`stack`은 인덱스에 의존하지만, **`melt`**는 평범한 컬럼을 그대로 long 형식으로 바꿔주는 더 직관적인 도구이다.

```python
pd.melt(
    df,
    id_vars=...,      # 그대로 유지할 컬럼 (식별자)
    value_vars=...,   # 녹일(녹여서 long으로 만들) 컬럼
    var_name=...,     # 변수명이 들어갈 새 컬럼 이름
    value_name=...,   # 값이 들어갈 새 컬럼 이름
)
```

### Ping 22 — 예제로 이해하기

In [ ]:
# Ping 22: melt의 기본 동작
wide = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'math': [90, 75, 85],
    'english': [85, 80, 95],
    'science': [78, 88, 92],
})
print('=== wide 형식 ===')
print(wide)
print()

# melt로 long 변환
long = wide.melt(
    id_vars='name',
    value_vars=['math', 'english', 'science'],
    var_name='subject',
    value_name='score',
)
print('=== long 형식 (melt 결과) ===')
print(long)

### **결과 해석**

- 3행 × 4열의 wide 데이터가 **9행 × 3열의 long 데이터**로 변환되었다.
- `name`은 **식별자**(id)로 유지되었고,
- `math`/`english`/`science` 컬럼은 **`subject`** 컬럼의 값으로 녹아내렸다.

> 이 long 형식이야말로 `seaborn`이나 `plotly`로 시각화할 때 **가장 다루기 쉬운 형태**이다. 뒤의 차시에서 그래프를 그릴 때 빈번히 사용된다.


### Ping 23 — `melt`로 시각화 준비

위의 long 데이터를 사용하면 한 줄로 깔끔한 그래프를 그릴 수 있다.

In [ ]:
# Ping 23: long 형식 → seaborn 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 (환경에 따라 생략 가능)
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8, 4))
sns.barplot(data=long, x='name', y='score', hue='subject')
plt.title('과목별 점수 비교')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

### **`hue` 파라미터의 비밀**

`seaborn`의 `hue=`가 작동하려면 그룹 구분 변수가 **별도 컬럼으로 존재**해야 한다. 즉 long 형식이 필수이다. 이 때문에 데이터 시각화의 첫 단계가 거의 항상 **`melt`로 시작**한다.

> **AI 연결**: 시계열 예측에서 다중 시리즈를 한 모델로 학습할 때(예: 여러 매장의 매출 예측), 입력 데이터를 long 형식으로 정렬하는 것이 표준이다. **PyTorch DataLoader**도 이 형태를 가장 자연스럽게 받아들인다.


## **4.4 `pivot` ↔ `melt` — 완전한 왕복 변환**

`melt`로 wide → long을 만든 뒤, `pivot`이나 `pivot_table`로 다시 wide로 되돌릴 수 있다. 이 두 연산은 **역연산** 관계이다.

### Ping 24 — 왕복 변환 검증

In [ ]:
# Ping 24: 왕복 변환
# wide → long → wide 변환이 원본을 복원하는지 확인

# 1) wide
wide = pd.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'math': [90, 75, 85],
    'english': [85, 80, 95],
})

# 2) wide → long
long = wide.melt(id_vars='name', var_name='subject', value_name='score')
print('=== long ===')
print(long)
print()

# 3) long → wide (pivot)
back_to_wide = long.pivot(index='name', columns='subject', values='score').reset_index()
back_to_wide.columns.name = None  # 컬럼 인덱스 이름 제거
print('=== 다시 wide로 ===')
print(back_to_wide)

### ✎ 개념 확인 8

`stack/unstack`과 `melt/pivot`의 차이는 무엇인가? 어떤 상황에 무엇을 쓰는가?

<details><summary>▶ 정답 보기</summary>

| 도구 | 작동 대상 | 결과 형태 |
|---|---|---|
| **`stack`/`unstack`** | **인덱스**(특히 MultiIndex)를 다룸 | Series 또는 DataFrame |
| **`melt`/`pivot`** | **일반 컬럼**을 다룸 | 항상 DataFrame |

**선택 기준**:

- `groupby` 결과에서 wide↔long 변환 → **`unstack`/`stack`** (이미 MultiIndex가 있으므로)
- 평범한 컬럼들을 long으로 녹이기 → **`melt`** (가장 직관적)
- long을 wide로 되돌리기 → **`pivot_table`** (중복이 있으면), `pivot`(없으면)

> 실무에서는 두 패러다임을 모두 익혀야 한다. 같은 결과를 두 방법 중 어느 쪽으로도 도달할 수 있는 것이 데이터 분석가의 자유도이다.

</details>


---

## **4.5 연습문제 (10문항)**

---

### **연습문제 4-1**

다음 wide 데이터를 long 형식으로 `melt`하라. `region`은 식별자로 유지한다.

In [ ]:
# 연습문제 4-1
sales = pd.DataFrame({
    'region': ['East', 'West', 'North'],
    '2022': [100, 150, 90],
    '2023': [120, 160, 110],
    '2024': [140, 180, 130],
})
print('=== 원본 (wide) ===')
print(sales)
print()

ans_4_1 = ____________________________________________
print('=== long ===')
print(ans_4_1)

<details><summary>▶ 정답 보기</summary>

```python
ans_4_1 = sales.melt(
    id_vars='region',
    value_vars=['2022', '2023', '2024'],
    var_name='year',
    value_name='sales',
)
print(ans_4_1)
```

`value_vars`를 생략하면 `id_vars`를 제외한 모든 컬럼이 자동으로 녹는다. 즉 `sales.melt(id_vars='region', var_name='year', value_name='sales')`도 같은 결과이다.

</details>

---

### **연습문제 4-2**

연습문제 4-1의 long 결과를 다시 wide로 되돌려라 (`pivot` 사용).

In [ ]:
# 연습문제 4-2
ans_4_2 = ____________________________________________
print(ans_4_2)

<details><summary>▶ 정답 보기</summary>

```python
ans_4_2 = ans_4_1.pivot(index='region', columns='year', values='sales')
print(ans_4_2)
```

`columns.name`이 `'year'`로 남아 있을 수 있다. 깔끔하게 정리하려면:

```python
ans_4_2 = ans_4_2.reset_index()
ans_4_2.columns.name = None
```

</details>

---

### **연습문제 4-3**

자치구별·방 유형별 평균 가격(`groupby`)을 wide 형식으로 만들어라 (`unstack` 사용).

In [ ]:
# 연습문제 4-3
ans_4_3 = ____________________________________________
print(ans_4_3.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_4_3 = (
    airbnb.groupby(['neighbourhood_group', 'room_type'])['price']
    .mean()
    .unstack()
)
print(ans_4_3.round(1))
```

</details>

---

### **연습문제 4-4**

연습문제 4-3의 결과를 다시 long으로 되돌려라 (`stack` 사용). 결과는 Series가 된다.

In [ ]:
# 연습문제 4-4
ans_4_4 = ____________________________________________
print(ans_4_4.round(1).head(10))

<details><summary>▶ 정답 보기</summary>

```python
ans_4_4 = ans_4_3.stack()
print(ans_4_4.round(1).head(10))
```

원래의 MultiIndex Series로 정확히 복원됨을 확인할 수 있다.

</details>

---

### **연습문제 4-5**

`covid` 데이터에서 **국가 × 월별 누적 사망자 수**의 wide 표를 만들어라.

In [ ]:
# 연습문제 4-5
ans_4_5 = ____________________________________________
print(ans_4_5.round(0).head(10))

<details><summary>▶ 정답 보기</summary>

```python
ans_4_5 = pd.pivot_table(
    covid,
    index='Country/Region',
    columns='Month',
    values='Deaths',
    aggfunc='sum',
    fill_value=0,
)
print(ans_4_5.round(0).head(10))
```

</details>

---

### **연습문제 4-6**

연습문제 4-5의 결과를 long 형식으로 되돌려라 (`reset_index` + `melt` 사용).

In [ ]:
# 연습문제 4-6
ans_4_6 = ____________________________________________
print(ans_4_6.head(10))

<details><summary>▶ 정답 보기</summary>

```python
ans_4_6 = ans_4_5.reset_index().melt(
    id_vars='Country/Region',
    var_name='Month',
    value_name='Deaths',
)
print(ans_4_6.head(10))
```

`pivot_table`의 결과는 인덱스가 그룹 키이므로, `reset_index()`로 일반 컬럼으로 되돌린 뒤 `melt`를 적용한다.

</details>

---

### **연습문제 4-7**

다음 wide 데이터에서 **`Q1`/`Q2`/`Q3`/`Q4`만 녹이고**, `name`과 `year`은 식별자로 유지하라.

In [ ]:
# 연습문제 4-7
quarters = pd.DataFrame({
    'name': ['Alice', 'Bob'],
    'year': [2024, 2024],
    'Q1': [10, 20],
    'Q2': [15, 25],
    'Q3': [20, 30],
    'Q4': [25, 35],
})
print('=== 원본 ===')
print(quarters)
print()

ans_4_7 = ____________________________________________
print('=== long ===')
print(ans_4_7)

<details><summary>▶ 정답 보기</summary>

```python
ans_4_7 = quarters.melt(
    id_vars=['name', 'year'],
    value_vars=['Q1', 'Q2', 'Q3', 'Q4'],
    var_name='quarter',
    value_name='value',
)
print(ans_4_7)
```

`id_vars`에 **여러 컬럼을 리스트로** 넘길 수 있다.

</details>

---

### **연습문제 4-8**

다음과 같은 wide 표를 long으로 변환하여 `seaborn.lineplot`으로 시각화하라.

In [ ]:
# 연습문제 4-8
import matplotlib.pyplot as plt
import seaborn as sns

scores = pd.DataFrame({
    'week': [1, 2, 3, 4, 5],
    'Alice': [60, 65, 70, 78, 85],
    'Bob':   [55, 60, 65, 72, 80],
    'Carol': [70, 75, 80, 88, 92],
})
print('=== 원본 ===')
print(scores)
print()

# (1) melt로 long 변환
long_scores = ____________________________________________

# (2) lineplot
plt.figure(figsize=(8, 4))
sns.lineplot(data=long_scores, x='week', y='score', hue='student', marker='o')
plt.title('주차별 학생 점수 추이')
plt.tight_layout()
plt.show()

<details><summary>▶ 정답 보기</summary>

```python
long_scores = scores.melt(
    id_vars='week',
    var_name='student',
    value_name='score',
)
```

이후 `sns.lineplot(data=long_scores, x='week', y='score', hue='student', marker='o')`로 그리면 학생별로 색이 다른 선 그래프가 나온다.

> 이 패턴은 **다중 시리즈 시각화의 표준**이다. 시계열 예측 모델의 학습 곡선, A/B 테스트 결과 비교 등에 두루 응용된다.

</details>

---

### **연습문제 4-9**

`airbnb`에서 자치구 × 방 유형 평균 가격을 만든 후, **자치구를 컬럼**으로 두는 wide 형식을 만들어라.

In [ ]:
# 연습문제 4-9
ans_4_9 = ____________________________________________
print(ans_4_9.round(1))

<details><summary>▶ 정답 보기</summary>

```python
ans_4_9 = (
    airbnb.groupby(['neighbourhood_group', 'room_type'])['price']
    .mean()
    .unstack(level='neighbourhood_group')
)
print(ans_4_9.round(1))
```

`unstack(level='neighbourhood_group')`로 **자치구 인덱스 층을 컬럼**으로 올린다.

</details>

---

### **연습문제 4-10**

다음 long 데이터를 wide로 변환하라. 단, 같은 (`student`, `subject`) 조합이 **여러 번 등장**하므로 평균을 사용하라.

In [ ]:
# 연습문제 4-10
records = pd.DataFrame({
    'student': ['A', 'A', 'B', 'B', 'A', 'B'],
    'subject': ['math', 'eng', 'math', 'eng', 'math', 'eng'],
    'score':   [90, 80, 70, 75, 85, 82],
})
print('=== 원본 (long, 중복 있음) ===')
print(records)
print()

ans_4_10 = ____________________________________________
print('=== wide (중복은 평균으로 처리) ===')
print(ans_4_10)

<details><summary>▶ 정답 보기</summary>

```python
ans_4_10 = pd.pivot_table(
    records,
    index='student',
    columns='subject',
    values='score',
    aggfunc='mean',
)
print(ans_4_10)
```

`pivot`은 중복이 있으면 에러를 내지만, `pivot_table`은 `aggfunc`으로 자동 집계한다. 실무에서 long → wide 변환 시 **항상 `pivot_table`을 기본값**으로 두는 이유이다.

</details>


---

# **1차시 마무리 — 종합 실습**

## **종합 과제 — Airbnb 데이터로 분석 보고서 만들기**

지금까지 배운 모든 도구(`groupby`, `crosstab`, `pivot_table`, `melt`/`unstack`)를 활용하여 짧은 분석 보고서를 작성하라.

### **요구 사항**

1. 자치구 × 방 유형별 **평균 가격 + 평균 리뷰 수 + 숙소 개수**를 보여주는 종합 표를 만든다.
2. 그 결과에서 **가장 비싸면서도 리뷰 수가 많은 자치구·방 유형 조합**을 찾아 출력한다.
3. 결과를 **히트맵**으로 시각화한다.


In [ ]:
# 종합 과제: 직접 작성해 보라
# (정답은 아래 토글)

# (1) 종합 표 만들기
summary = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values=['price', 'number_of_reviews'],
    aggfunc=['mean', 'count'],
    fill_value=0,
)

# (2) 평균 가격 부분만 추출
avg_price = pd.pivot_table(
    airbnb,
    index='neighbourhood_group',
    columns='room_type',
    values='price',
    aggfunc='mean',
)

# (3) 가장 비싼 조합 찾기
import numpy as np
flat = avg_price.stack()  # MultiIndex Series
top_combo = flat.idxmax()
top_price = flat.max()
print(f'가장 비싼 조합: {top_combo}, 평균 가격 ${top_price:.0f}')

# (4) 히트맵
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 5))
sns.heatmap(avg_price, annot=True, fmt='.0f', cmap='YlOrRd', cbar_kws={'label': '평균 가격 ($)'})
plt.title('자치구 × 방 유형별 평균 가격', fontsize=13)
plt.tight_layout()
plt.show()

<details><summary>▶ 종합 과제 풀이 해설</summary>

**핵심 흐름**

1. `pivot_table`로 다축 요약을 만든다.
2. `stack()`으로 MultiIndex Series로 펴서 `idxmax()`로 최댓값 위치를 찾는다.
3. `seaborn.heatmap`으로 wide 표를 시각화한다.

**점검 포인트**

- `cmap='YlOrRd'` 같은 색상 맵은 **순차형**(sequential)이라 양수 데이터에 적합하다.
- 음수가 섞이면 `cmap='RdBu_r'` 같은 **발산형**(diverging)이 더 적절하다.
- `annot=True`로 셀에 숫자를 표시하면 가독성이 크게 향상된다.

</details>

---

## **1차시 핵심 정리**

| 주제 | 핵심 메서드 | 한 줄 요약 |
|---|---|---|
| **그룹별 집계** | `groupby + agg` | 그룹 키별로 통계량 계산 |
| **그룹별 변환** | `groupby + transform` | 그룹 통계량을 원본 길이로 매핑 |
| **그룹 필터링** | `groupby + filter` | 조건을 만족하는 그룹만 남기기 |
| **빈도 교차표** | `crosstab` | 두 범주의 빈도/비율 표 |
| **다축 요약** | `pivot_table` | 다중 인덱스/컬럼/값/집계의 통합 |
| **wide → long** | `melt`, `stack` | 시각화·모델링용 long 변환 |
| **long → wide** | `pivot`, `unstack` | 보고서·인쇄용 wide 변환 |

---

## **다음 차시 예고 — EDA 시각화 마스터 (3시간)**

다음 차시에서는 다음을 배운다.

- **시각화 의사결정 프레임워크** — 어떤 상황에 어떤 그래프?
- **분포** 그래프: 히스토그램, KDE, boxplot, violinplot
- **관계** 그래프: 산점도, pairplot, heatmap, 회귀선
- **비교** 그래프: 막대, grouped bar, 누적 막대
- **인터랙티브** 시각화: `plotly`로 반응형 그래프 만들기
- **종합 EDA 사례**: Airbnb 데이터 전체 EDA 워크플로우

> 1차시에서 배운 **wide↔long 변환**이 다음 차시 **시각화의 입력 데이터를 만드는 첫 단계**가 된다. 두 차시는 한 흐름이다.

---

**1차시 끝 — 수고했다.**
